<!-- colab-badge -->
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aibizx/python-primer-notebooks/blob/main/20-evaluation.ipynb)

_Part of the [AI/Biz books](https://www.ai.biz/books/python-primer/) collection._

# Chapter 20 — Evaluation, and How It Lies to You

Companion to [the chapter](https://www.ai.biz/books/python-primer/evaluation/).

Every form of leakage in this notebook produces a *better* score. That is what makes it dangerous.


In [ ]:
import numpy as np, pandas as pd
from sklearn.model_selection import (train_test_split, cross_val_score,
    cross_validate, StratifiedKFold, GroupKFold, TimeSeriesSplit)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
rng = np.random.default_rng(0)


In [ ]:
n = 1500
X = pd.DataFrame(rng.normal(size=(n, 5)), columns=[f'f{i}' for i in range(5)])
y = pd.Series((X.f0 * 0.9 + X.f1 * 0.6 + rng.normal(0, 1, n) > 0).astype(int))
X.loc[rng.choice(n, 200, replace=False), 'f2'] = np.nan
cv = StratifiedKFold(5, shuffle=True, random_state=0)
print('churn rate:', y.mean().round(3))


## 1. Leakage from preprocessing before the split


In [ ]:
imp, sc = SimpleImputer(strategy='mean'), StandardScaler()
X_leaked = pd.DataFrame(sc.fit_transform(imp.fit_transform(X)), columns=X.columns)
leaked = cross_val_score(LogisticRegression(), X_leaked, y, cv=cv, scoring='roc_auc')

pipe = Pipeline([('impute', SimpleImputer(strategy='mean')),
                 ('scale', StandardScaler()),
                 ('model', LogisticRegression())])
honest = cross_val_score(pipe, X, y, cv=cv, scoring='roc_auc')

print(f'leaked : {leaked.mean():.5f}')
print(f'honest : {honest.mean():.5f}')
print(f'gap    : {leaked.mean()-honest.mean():+.5f}  <- small here, always flattering')


The gap grows with imputation, target encoding and feature selection. You can never see it, which is why it must be made structurally impossible.


## 2. Duplicate rows across folds


In [ ]:
# 300 customers, each appearing ~5 times
groups = rng.integers(0, 300, n)
Xg = X.copy(); Xg['customer_effect'] = groups / 300.0
yg = pd.Series(((groups % 2 == 0).astype(float)*1.5 + rng.normal(0,1,n) > 0.5).astype(int))

naive = cross_val_score(pipe, Xg, yg, cv=StratifiedKFold(5, shuffle=True, random_state=0),
                        scoring='roc_auc').mean()
grouped = cross_val_score(pipe, Xg, yg, cv=GroupKFold(5), groups=groups,
                          scoring='roc_auc').mean()
print(f'random folds (same customer in train AND test): {naive:.3f}')
print(f'GroupKFold  (customer in one fold only)      : {grouped:.3f}')


## 3. Temporal leakage


In [ ]:
dates = pd.date_range('2026-01-01', periods=n, freq='h')
trend = np.linspace(0, 2, n)
Xt = X.copy()
yt = pd.Series((X.f0*0.5 + trend + rng.normal(0,1,n) > 1).astype(int))

shuffled = cross_val_score(pipe, Xt, yt, cv=StratifiedKFold(5, shuffle=True, random_state=0),
                           scoring='roc_auc').mean()
temporal = cross_val_score(pipe, Xt, yt, cv=TimeSeriesSplit(5), scoring='roc_auc').mean()
print(f'random shuffle (trains on the future): {shuffled:.3f}')
print(f'TimeSeriesSplit (past predicts future): {temporal:.3f}')


## 4. Train vs validation score is a free diagnostic


In [ ]:
from sklearn.ensemble import RandomForestClassifier

for label, model in [('underfit  (max_depth=1)', RandomForestClassifier(max_depth=1, n_estimators=20, random_state=0)),
                     ('healthy   (max_depth=4)', RandomForestClassifier(max_depth=4, n_estimators=50, random_state=0)),
                     ('overfit   (no limit)   ', RandomForestClassifier(n_estimators=50, random_state=0))]:
    p = Pipeline([('impute', SimpleImputer()), ('model', model)])
    s = cross_validate(p, X, y, cv=cv, scoring='roc_auc', return_train_score=True)
    print(f'{label}: train {s["train_score"].mean():.3f} | val {s["test_score"].mean():.3f}'
          f' | gap {s["train_score"].mean()-s["test_score"].mean():+.3f}')


## 5. Accuracy is the wrong metric on imbalanced data


In [ ]:
from sklearn.metrics import (accuracy_score, roc_auc_score, average_precision_score,
                             precision_score, recall_score, confusion_matrix)

y_rare = pd.Series((rng.random(n) < 0.01).astype(int))    # 1% positive
always_negative = np.zeros(n)
print(f'accuracy of predicting "never": {accuracy_score(y_rare, always_negative):.3f}')
print('99% accurate and completely useless.')


## 6. The threshold is a separate business decision


In [ ]:
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, stratify=y, random_state=0)
pipe.fit(Xtr, ytr)
proba = pipe.predict_proba(Xte)[:, 1]

print(f'{"threshold":>10} {"precision":>10} {"recall":>8} {"flagged":>8}')
for t in [0.1, 0.3, 0.5, 0.7, 0.9]:
    pred = (proba >= t).astype(int)
    print(f'{t:>10} {precision_score(yte, pred, zero_division=0):>10.3f}'
          f' {recall_score(yte, pred):>8.3f} {pred.sum():>8}')
print()
print('0.5 is a default, not an answer. Pick it from the costs.')


## 7. Calibration: ranking well is not the same as being right


In [ ]:
from sklearn.calibration import calibration_curve
prob_true, prob_pred = calibration_curve(yte, proba, n_bins=8)
print(f'{"predicted":>10} {"observed":>10}')
for p, t in zip(prob_pred, prob_true):
    print(f'{p:>10.3f} {t:>10.3f}')
print()
print('If these diverge, a probability multiplied by money will be wrong.')
print(f'AUC is blind to this: {roc_auc_score(yte, proba):.3f}')


## 8. Always beat a dumb baseline


In [ ]:
base = cross_val_score(DummyClassifier(strategy='prior'), X, y, cv=cv, scoring='roc_auc').mean()
model = honest.mean()
print(f'baseline : {base:.3f}   <- coin flip by construction')
print(f'model    : {model:.3f}')
print(f'lift     : {model-base:+.3f}')
print()
print('Three lines. It has saved more projects than any modelling technique.')


## Try it yourself

1. Add a feature equal to `y` plus noise and watch the AUC hit 0.99. That is what leakage looks like.
2. Compare `roc_auc` against `average_precision` on the 1% imbalanced target.
3. Wrap the model in `CalibratedClassifierCV` and re-check the calibration table.
